# E3 — coPILOT

[Artículo](https://doi.org/10.1007/s11634-026-00683-8). Experimento solicitado de la búsqueda Scopus.

Se usa sin modificar el código de los autores de wPILOT,
commit4ae908f4f1bd9ec93f1eeaebde958d6dd7ec5943. Se entrenan exactamente dos
árboles: `max_n_estimators=2`, `stop_early=False` y predicción `Average`.
El segundo árbol pondera errores del primero; no genera datos sintéticos.

Mediana numérica, moda y codificación ordinal de categorías dentro del fold.
Los siete índices categóricos se pasan explícitamente al algoritmo para
sus divisiones nominales, no como pendientes numéricas. Sin escalado/log1p
externos. Grilla18: profundidad2/4/6 × hoja10/30 × alpha.2/.5/1;
min_sample_split=2×hoja y max_model_depth30. Se conserva la truncación interna
del algoritmo (basada solo en fit), sin recorte externo de predicciones.
El código original convierte los grados de libertad del BIC a enteros;
esa discretización de alpha se conserva y no reproduce aritmética fraccional
idealizada. No se usan pesos por covariate shift ni covariables de test.

## Protocolo compartido

Misma fuente real `Data/Datos_Brutos.csv`, reglas de negocio y15 predictores.
Objetivo `nota_primer_parcial`; nota_final, pidm, periodo e índice de fila no
son entradas. Sin aumentación ni eliminación de outliers válidos.
Train1309 filas/1293 estudiantes; test202530:188 estudiantes, sin solapamiento.

Cinco folds externos canónicos por estudiante, cinco internos en cada búsqueda
y cinco internos para elegir parámetros finales. RMSE medio interno selecciona
la configuración. Cada preprocesador y modelo se ajusta solo en su fit.
La selección final se guarda antes de predecir test. Mismas funciones del tablero:
RMSE, MAE, MAPE(%) y R²; métricas sin ponderación, escala original.
Media/SD entre folds, OOF reunido y test temporal se informan por separado.

Se guardan grillas, auditorías, hashes, OOF y predicciones test. El código puede
reanudar cálculos de igual procedencia; si cambia la procedencia exige una
carpeta nueva. Pesos/estimadores quedan locales en modelos_preentrenados/scopus.
No se reemplazan los modelos del dashboard ni se reescriben informes anteriores.

In [1]:
from pathlib import Path
import sys
BOOK=Path.cwd() if (Path.cwd()/'entregable3.py').exists() else Path.cwd()/'Precalculo'
sys.path.insert(0,str(BOOK.resolve()))
import pandas as pd
from IPython.display import display
from evaluar_scopus_e3 import run,compare,ROOT

report=run('copilot')
display(pd.DataFrame(report['cv']).round(6))
print('Parámetros finales:',report['best_params'])

,metrica,media_cv,sd_cv,OOF_agrupado
0,RMSE,0.859075,0.026887,0.859505
1,MAE,0.683411,0.025023,0.683426
2,MAPE (%),25.387139,1.354007,25.387270
3,R²,0.405093,0.057060,0.408545


Parámetros finales: {'alpha': 0.5, 'depth': 6, 'leaf': 30}


## Test temporal y auditoría

El test no selecciona hiperparámetros.

In [2]:
display(pd.DataFrame([report['test']]).round(6))
display(pd.DataFrame(report['fold_audit']))

,RMSE,MAE,MAPE (%),R²
0,0.970603,0.793968,30.540858,0.374472


,fold,best_params,inner_rmse,fit_rows,validation_rows,shared_students,seconds
0,1,"{'alpha': 1.0, 'depth': 4, 'leaf': 10}",0.867170,1047,262,0,105.928352
1,2,"{'alpha': 1.0, 'depth': 4, 'leaf': 10}",0.829613,1047,262,0,51.528706
2,3,"{'alpha': 0.2, 'depth': 4, 'leaf': 30}",0.830855,1047,262,0,47.829180
3,4,"{'alpha': 0.2, 'depth': 6, 'leaf': 30}",0.851140,1047,262,0,47.987200
4,5,"{'alpha': 1.0, 'depth': 4, 'leaf': 30}",0.838624,1048,261,0,51.540165


## Comparación conjunta

Tras completar los cuatro modelos, consultar precalculo_scopus_e3.ipynb y resultados/e3/scopus/comparacion.json.